# 🔬 Optimizer Trajectory Comparison on Non-Convex Surfaces

In this notebook, we compare the convergence dynamics of **SGD**, **SGD with Momentum**, **RMSprop**, and **Adam** on the famous non-convex **Rosenbrock function** (the "banana function"):

$$f(x, y) = (a - x)^2 + b(y - x^2)^2 \quad \text{where } a=1, b=100$$

The global minimum is located inside a long, curved, narrow parabolic valley at $(x^*, y^*) = (1, 1)$, where $f(1, 1) = 0$.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure local src is in path
src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from autograd import Value
from optimizers import SGD, SGDMomentum, RMSprop, Adam

### Defining the Optimization Experiment
We initialize all optimizers from the same difficult starting point $(-1.5, 2.0)$ and record their 2D trajectories.

In [ ]:
def optimize_rosenbrock(opt_class, lr=0.002, steps=1000, start_point=(-1.5, 2.0)):
    x = Value(start_point[0])
    y = Value(start_point[1])
    opt = opt_class([x, y], lr=lr)
    
    trajectory = [(x.data, y.data)]
    
    for _ in range(steps):
        opt.zero_grad()
        # f(x, y) = (1 - x)^2 + 100 * (y - x^2)^2
        loss = (Value(1.0) - x)**2 + Value(100.0) * (y - x**2)**2
        loss.backward()
        opt.step()
        trajectory.append((x.data, y.data))
        
    return np.array(trajectory)

trajectories = {
    "SGD": optimize_rosenbrock(SGD, lr=0.001, steps=800),
    "SGDMomentum": optimize_rosenbrock(SGDMomentum, lr=0.001, steps=800),
    "RMSprop": optimize_rosenbrock(RMSprop, lr=0.01, steps=800),
    "Adam": optimize_rosenbrock(Adam, lr=0.02, steps=800),
}

print("Optimization simulations complete!")
for name, traj in trajectories.items():
    final_pos = traj[-1]
    dist_to_opt = np.sqrt((final_pos[0] - 1.0)**2 + (final_pos[1] - 1.0)**2)
    print(f"{name:12s} Final: ({final_pos[0]:.3f}, {final_pos[1]:.3f}) | Distance to (1,1): {dist_to_opt:.3f}")

### Plotting Contour Surface and Convergence Paths

In [ ]:
# Generate Rosenbrock contour grid
x_vals = np.linspace(-2.0, 2.0, 400)
y_vals = np.linspace(-1.0, 3.0, 400)
X_grid, Y_grid = np.meshgrid(x_vals, y_vals)
Z_grid = (1.0 - X_grid)**2 + 100.0 * (Y_grid - X_grid**2)**2

plt.figure(figsize=(10, 7))
plt.contour(X_grid, Y_grid, np.log1p(Z_grid), levels=30, cmap='viridis', alpha=0.6)
plt.plot(1, 1, 'r*', markersize=16, label='Global Minimum (1, 1)')

colors = {'SGD': 'blue', 'SGDMomentum': 'orange', 'RMSprop': 'purple', 'Adam': 'red'}
for name, traj in trajectories.items():
    plt.plot(traj[:, 0], traj[:, 1], label=name, color=colors[name], linewidth=2)

plt.title('Optimizer Trajectories on Rosenbrock Loss Surface')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()